In [3]:
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm

from src.parameters import MAX_PRECIP_MM, MIN_TEMP_C, MAX_TEMP_C

tqdm.pandas()

/Users/Patron/Documents/cs524/.gamspy_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
sundays_df = pd.read_csv('data/processed/sundays_2026.csv')
circuits_df = pd.read_csv('data/processed/circuits.csv')
sundays_df.head()

,race_date,week_num,month
0,2026-01-04,1,1
1,2026-01-11,2,1
2,2026-01-18,3,1
3,2026-01-25,4,1
4,2026-02-01,5,2


In [5]:
# Get season start and end dates

start_date = sundays_df[sundays_df['month'] == 3]['race_date'].values[0]
end_date = sundays_df[sundays_df['month'] == 12]['race_date'].values[0]

start_week = sundays_df[sundays_df['month'] == 3]['week_num'].values[0]

print(f"Season start date: {start_date} (week: {start_week}), end date: {end_date}")


Season start date: 2026-03-01 (week: 9), end date: 2026-12-06


In [6]:
mask = (sundays_df['race_date'] >= start_date) & (sundays_df['race_date'] <= end_date)

race_weekends_df = sundays_df.loc[mask].reset_index(drop=True)
race_weekends_df['week_num'] = race_weekends_df['week_num'] - start_week + 1

race_weekends_df.head()

,race_date,week_num,month
0,2026-03-01,1,3
1,2026-03-08,2,3
2,2026-03-15,3,3
3,2026-03-22,4,3
4,2026-03-29,5,3


In [35]:
API_KEY = "U2333EX6VCZBN2545ACQYFPV4"
# API_KEY = "9C84VDUB86RH2E9ATA5HZAWAV"
# API_KEY = "L6P4AVRPDA2YQSCVCUZG8STPG"

def get_weather_history(row, coords, years=5):
    base_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline"
    params = {
        "key": API_KEY,
        "include": "days",
        "contentType": "json"
    }
    temps = []
    precips = []
    date = row['race_date']

    for year_offset in range(1, years+1):
        year = int(date[:4]) - year_offset
        new_date = f"{year}{date[4:]}"
        url = f"{base_url}/{coords}/{new_date}"
        response = requests.get(url, params=params)

        try:
            res = response.json()
            day_data = res['days'][0]
            temps.append(day_data['temp'])
            precips.append(day_data['precip'])

        except Exception as e:
            print(f"Error fetching data for {coords} on {new_date}: {e}")
            exit(1)
        time.sleep(1)  # To respect API rate limits

    # Return average temperature and precipitation with 2 decimal places
    return pd.Series([round(np.mean(temps), 2), round(np.mean(precips), 2)])

In [40]:
for index, circuit in circuits_df.iterrows():


    circuit_id = circuit['circuit_id']
    coords = f"{circuit['latitude']},{circuit['longitude']}"
    tobedone = ['QAT', 'ABD']

    print(f"Processing circuit: {circuit_id} at {coords} [{index+1}/{len(circuits_df)}]")
    if circuit_id in tobedone:
        race_weekends_df[[f'{circuit_id}_avg_temp', f'{circuit_id}_avg_precip']] = race_weekends_df.progress_apply(lambda row: get_weather_history(row, coords), axis=1)

display(race_weekends_df.head())

Processing circuit: BAH at 26.0316553,50.5145563 [1/24]
Processing circuit: SAU at 21.5504432,39.1742363 [2/24]
Processing circuit: AUS at -37.8142454,144.9631732 [3/24]
Processing circuit: JPN at 34.8817102,136.5836516 [4/24]
Processing circuit: CHN at 31.2312707,121.4700152 [5/24]
Processing circuit: MIA at 25.7741566,-80.1935973 [6/24]
Processing circuit: MON at 43.7402961,7.426559 [7/24]
Processing circuit: CAN at 45.5031824,-73.5698065 [8/24]
Processing circuit: ESP at 41.3825802,2.177073 [9/24]
Processing circuit: MAD at 40.416782,-3.703507 [10/24]
Processing circuit: AUT at 47.2122736,14.7857412 [11/24]
Processing circuit: GBR at 52.0917705,-1.0260194 [12/24]
Processing circuit: HUN at 47.5986636,19.2384215 [13/24]
Processing circuit: BEL at 50.3942409,5.9310335 [14/24]
Processing circuit: NED at 52.3719838,4.5302209 [15/24]
Processing circuit: MONZ at 45.6395418,9.2788304 [16/24]
Processing circuit: SIN at 1.2899175,103.8519072 [17/24]
Processing circuit: AZE at 40.3755885,49.8

100%|██████████| 41/41 [04:06<00:00,  6.02s/it]


Processing circuit: ABD at 24.4538352,54.3774014 [24/24]


100%|██████████| 41/41 [04:02<00:00,  5.92s/it]


,race_date,week_num,month,BAH_avg_temp,BAH_avg_precip,SAU_avg_temp,SAU_avg_precip,AUS_avg_temp,AUS_avg_precip,JPN_avg_temp,...,USA_LVG_avg_temp,USA_LVG_avg_precip,MEX_avg_temp,MEX_avg_precip,BRA_avg_temp,BRA_avg_precip,QAT_avg_temp,QAT_avg_precip,ABD_avg_temp,ABD_avg_precip
0,2026-03-01,1,3,65.98,0.00,78.98,0.00,66.40,0.01,48.24,...,56.20,0.05,64.66,0.00,75.14,0.30,68.38,0.0,70.38,0.00
1,2026-03-08,2,3,72.10,0.01,79.56,0.00,65.62,0.05,45.22,...,56.44,0.00,65.00,0.00,74.30,0.65,74.16,0.0,79.26,0.01
2,2026-03-15,3,3,72.04,0.00,80.76,0.01,69.16,0.01,50.70,...,56.28,0.06,63.44,0.05,74.68,0.01,74.16,0.0,76.74,0.00
3,2026-03-22,4,3,74.32,0.00,78.40,0.00,63.60,0.03,50.52,...,61.94,0.01,65.50,0.02,71.50,0.06,76.64,0.0,78.66,0.00
4,2026-03-29,5,3,71.66,0.00,84.56,0.00,63.66,0.06,55.46,...,63.82,0.00,64.04,0.19,73.26,0.02,74.22,0.0,75.94,0.00


In [41]:
display(race_weekends_df[race_weekends_df['month'].isin([10, 11, 12])])
race_weekends_df.to_csv('data/processed/climate.csv', index=False)  

,race_date,week_num,month,BAH_avg_temp,BAH_avg_precip,SAU_avg_temp,SAU_avg_precip,AUS_avg_temp,AUS_avg_precip,JPN_avg_temp,...,USA_LVG_avg_temp,USA_LVG_avg_precip,MEX_avg_temp,MEX_avg_precip,BRA_avg_temp,BRA_avg_precip,QAT_avg_temp,QAT_avg_precip,ABD_avg_temp,ABD_avg_precip
31,2026-10-04,32,10,89.84,0.0,88.52,0.00,58.82,0.20,71.80,...,78.88,0.00,62.18,0.13,66.18,0.05,90.66,0.00,91.46,0.00
32,2026-10-11,33,10,88.14,0.0,88.76,0.00,57.32,0.01,67.60,...,75.28,0.00,61.84,0.01,69.28,0.04,88.64,0.01,89.20,0.00
33,2026-10-18,34,10,86.58,0.0,88.50,0.00,59.02,0.03,65.24,...,69.34,0.00,61.62,0.02,65.32,0.21,88.12,0.00,88.28,0.01
34,2026-10-25,35,10,83.92,0.0,86.68,0.00,55.20,0.06,60.58,...,67.58,0.00,62.92,0.07,68.88,0.12,85.06,0.00,86.48,0.01
35,2026-11-01,36,11,82.96,0.0,87.30,0.00,56.64,0.03,60.98,...,64.72,0.00,60.68,0.00,66.68,0.12,83.92,0.00,85.06,0.00
36,2026-11-08,37,11,80.20,0.0,85.42,0.00,60.64,0.15,58.18,...,62.38,0.02,62.30,0.07,67.72,0.36,82.84,0.00,82.96,0.00
37,2026-11-15,38,11,78.02,0.0,83.04,0.03,56.14,0.05,55.78,...,58.18,0.09,59.14,0.00,70.58,0.20,79.40,0.00,81.86,0.00
38,2026-11-22,39,11,76.68,0.0,83.44,0.00,63.46,0.01,56.40,...,53.78,0.00,58.82,0.33,71.84,0.11,78.32,0.00,80.36,0.01
39,2026-11-29,40,11,72.24,0.0,83.00,0.00,64.92,0.02,50.94,...,51.04,0.01,60.20,0.04,74.36,0.34,74.10,0.00,77.38,0.00
40,2026-12-06,41,12,72.04,0.0,82.16,0.02,66.14,0.01,49.48,...,54.50,0.00,58.36,0.00,72.36,0.04,73.68,0.00,76.06,0.00
